# getitrack: Real-Video Multi-Object Tracking

Track objects in real videos with **RF-DETR** detection and getitrack's **ByteTrack**.
Runs end to end on Google Colab (or locally).

Pipeline: **frame -> RF-DETR -> `Detections` -> ByteTrack -> annotated video**. The
detector is wrapped as a getitrack `DetectionAdapter`, so any detector plugs in the same way.

## 1. Install

In [ ]:
!pip install -q uv
!uv pip install -q --system rfdetr 'getitrack @ git+https://github.com/omkar-334/otx.git@getitrack-demo#subdirectory=libraries/getitrack'

## 2. Download demo videos

Three short clips (bicycles/motorcycles, airplanes, apples) from the project release.

In [ ]:
import urllib.request
from pathlib import Path

Path('videos').mkdir(exist_ok=True)
BASE = 'https://github.com/omkar-334/otx/releases/download/demo-videos'
for name in ['bikes-1', 'jets-1', 'apples']:
    dst = Path('videos') / f'{name}.mp4'
    if not dst.exists():
        urllib.request.urlretrieve(f'{BASE}/{name}.mp4', dst)
    print('ready', dst)

## 3. Detector adapter

`RFDETRAdapter` wraps RF-DETR behind getitrack's `DetectionAdapter` interface,
emitting `Detections` for the requested COCO classes.

In [ ]:
import cv2
import numpy as np
from rfdetr import RFDETRNano

from getitrack.adapters import DetectionAdapter
from getitrack.core.detection import Detections
from getitrack.utils import COCO_CLASSES

CLASS_SETS = {'bikes': [2, 4], 'jets': [5], 'apples': [53]}  # COCO ids


class RFDETRAdapter(DetectionAdapter):
    def __init__(self, class_ids, score_thresh=0.4):
        self._wanted = set(class_ids)
        self._score_thresh = score_thresh
        self._model = RFDETRNano()

    @property
    def class_names(self):
        return COCO_CLASSES

    def detect(self, frame_bgr, frame_id):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        det = self._model.predict(rgb, threshold=self._score_thresh)
        if det.class_id is None or len(det) == 0:
            return Detections.create_empty(frame_id)
        keep = np.array([c in self._wanted for c in det.class_id])
        if not keep.any():
            return Detections.create_empty(frame_id)
        return Detections(
            bboxes=det.xyxy[keep].astype(np.float32),
            scores=det.confidence[keep].astype(np.float32),
            class_ids=det.class_id[keep].astype(np.int64),
            frame_id=frame_id,
        )

## 4. Tracking pipeline

For each frame: detect, `tracker.update`, then draw the tracked boxes and ids.

In [ ]:
from pathlib import Path

from getitrack import BaseTracker, ByteTrackConfig, TrackAnnotator, to_h264
from getitrack.io import VideoReader, VideoWriter


def track_video(video, class_ids, output, adapter, width=960, max_frames=0):
    video, output = Path(video), Path(output)
    tracker = BaseTracker.from_config(ByteTrackConfig())
    annotator = TrackAnnotator(show_score=True, class_names=COCO_CLASSES)
    with VideoReader(video) as reader:
        size = (width, round(reader.height * width / reader.width))
        raw = output.with_name(f'{output.stem}.raw.mp4')
        with VideoWriter(raw, fps=reader.fps or 30.0, frame_size=size) as writer:
            for frame_id, frame in enumerate(reader):
                if max_frames and frame_id >= max_frames:
                    break
                resized = cv2.resize(frame, size)
                tracked = tracker.update(adapter.detect(resized, frame_id))
                writer.write(annotator.annotate(resized, tracked))
    final = to_h264(raw, output)
    raw.unlink(missing_ok=True)
    print(f'{video.name}: {writer.frames_written} frames -> {final}')
    return final

## 5. Run and watch

Track each clip for the class it contains and show the annotated result inline.

In [ ]:
from IPython.display import Video, display

Path('results').mkdir(exist_ok=True)
for name, src in [('bikes', 'videos/bikes-1.mp4'), ('jets', 'videos/jets-1.mp4'), ('apples', 'videos/apples.mp4')]:
    adapter = RFDETRAdapter(CLASS_SETS[name])
    out = track_video(src, CLASS_SETS[name], f'results/{name}_tracked.mp4', adapter)
    display(Video(str(out), embed=True, width=720))

## Track your own video

Point `track_video` at any file and pass the COCO class ids to follow
(see `getitrack.utils.COCO_CLASSES` for the full list).

In [ ]:
# adapter = RFDETRAdapter([3, 6, 8])  # car, bus, truck
# out = track_video('videos/my_video.mp4', [3, 6, 8], 'results/my_tracked.mp4', adapter)
# Video(str(out), embed=True, width=720)